# Memento Design Pattern 

explained using the classic Text Editor (Undo/Redo) example.

#### The Concept
The Memento Pattern captures and externalizes an object's internal state so that the object can be restored to this state later. 

**Crucial Point**: It allows you to save the state **without violating encapsulation**. The "Caretaker" (History Manager) holds the saved state (Memento) but cannot read or tamper with its contents. Only the "Originator" (Editor) can read it.

**Analogy**: A Video Game Save Point.
- **Originator**: The Game Logic (Current Level, Health, Inventory).
- **Memento**: The Save File (Binary blob).
- **Caretaker**: The Memory Card (Holds the file but doesn't know how to play the game).

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we explicitly define a `Memento` class. The `Originator` creates the memento, and the `Caretaker` stores it.

#### THE MEMENTO (The Snapshot)

In [1]:
class TextMemento:
    """
    Ideally, this class is immutable. 
    It stores the state of the Editor at a specific point in time.
    """
    def __init__(self, content: str):
        self._content = content

    def get_saved_content(self) -> str:
        return self._content

#### THE ORIGINATOR (The Editor)

In [2]:
class TextEditor:
    def __init__(self):
        self._content = ""

    def type_words(self, words: str):
        self._content += " " + words
        print(f"Current Content: '{self._content.strip()}'")

    def save(self) -> TextMemento:
        """Creates a snapshot."""
        print("   (Saving state...)")
        return TextMemento(self._content)

    def restore(self, memento: TextMemento):
        """Restores state from a snapshot."""
        self._content = memento.get_saved_content()
        print(f"   (Restored to: '{self._content.strip()}')")

#### THE CARETAKER (History Manager)

In [3]:
from typing import List

class History:
    """
    Keeps track of Mementos. 
    It knows WHEN to save/undo, but doesn't know WHAT is inside.
    """
    def __init__(self):
        self._history: List[TextMemento] = []

    def push(self, memento: TextMemento):
        self._history.append(memento)

    def pop(self) -> TextMemento:
        if not self._history:
            return None
        return self._history.pop()

#### CLIENT CODE

In [4]:
def main():
    editor = TextEditor()
    history = History()

    # 1. Type something
    editor.type_words("Hello")
    history.push(editor.save()) # SAVE 1

    # 2. Type more
    editor.type_words("World")
    history.push(editor.save()) # SAVE 2

    # 3. Mistake
    editor.type_words("OOPS!!!")
    
    # 4. Undo (Restore Save 2)
    print("\n--- UNDO 1 ---")
    editor.restore(history.pop())

    # 5. Undo again (Restore Save 1)
    print("--- UNDO 2 ---")
    editor.restore(history.pop())

if __name__ == "__main__":
    main()

Current Content: 'Hello'
   (Saving state...)
Current Content: 'Hello World'
   (Saving state...)
Current Content: 'Hello World OOPS!!!'

--- UNDO 1 ---
   (Restored to: 'Hello World')
--- UNDO 2 ---
   (Restored to: 'Hello')


## The Pythonic Way

In Python, we often don't need a specific `Memento` class. We can use python's built-in serialization (like pickle) or `deepcopy` to capture state. Because Python objects are dynamic, "State" is just the `__dict__` attribute. We can snapshot the entire object dictionary or deep-copy specific attributes.

#### THE ORIGINATOR

In [5]:
from typing import List
from dataclasses import dataclass

@dataclass
class GameCharacter:
    name: str
    health: int
    level: int
    inventory: List[str]

    def take_damage(self, amount):
        self.health -= amount
        print(f"💥 {self.name} took {amount} dmg (Health: {self.health})")

    def level_up(self):
        self.level += 1
        print(f"🆙 {self.name} reached Level {self.level}!")

    # PYTHONIC SAVE: Return a Deep Copy of state
    def save_state(self):
        # We don't need a wrapper class. The state itself is the memento.
        return copy.deepcopy(self)

    # PYTHONIC RESTORE: Copy attributes back
    def restore_state(self, memento: 'GameCharacter'):
        self.health = memento.health
        self.level = memento.level
        self.inventory = memento.inventory
        print(f"🔄 State Restored: Lv{self.level}, HP{self.health}")

#### CLIENT CODE (The Caretaker)

In [8]:
import copy

def main():
    # 1. Start Game
    hero = GameCharacter(name="Arthur", health=100, level=1, inventory=["Sword"])
    
    # Using a simple List as our Caretaker Stack
    save_points = []

    # 2. Save Game (Checkpoint 1)
    print("--- Save Point 1 ---")
    save_points.append(hero.save_state())

    # 3. Progress
    hero.level_up()
    hero.take_damage(20)
    
    # 4. Save Game (Checkpoint 2)
    print("--- Save Point 2 ---")
    save_points.append(hero.save_state())

    # 5. Disaster happens!
    hero.take_damage(999) # Hero dies

    # 6. Load Last Save
    print("\n--- Loading Last Save... ---")
    last_save = save_points.pop()
    hero.restore_state(last_save)

    # 7. Load Previous Save
    print("\n--- Loading Previous Save... ---")
    old_save = save_points.pop()
    hero.restore_state(old_save)

if __name__ == "__main__":
    main()

--- Save Point 1 ---
🆙 Arthur reached Level 2!
💥 Arthur took 20 dmg (Health: 80)
--- Save Point 2 ---
💥 Arthur took 999 dmg (Health: -919)

--- Loading Last Save... ---
🔄 State Restored: Lv2, HP80

--- Loading Previous Save... ---
🔄 State Restored: Lv1, HP100


## Key Differences

| Feature            | Classic OOP                                                         | Pythonic                                                         |
|--------------------|---------------------------------------------------------------------|------------------------------------------------------------------|
| **State Storage**  | Custom `Memento` class encapsulating object fields.                 | `copy.deepcopy()` or `pickle` to snapshot the object.           |
| **Complexity**     | High — requires three roles (Originator, Caretaker, Memento).       | Low — originator creates a copy; client or list acts as caretaker. |
| **Encapsulation**  | Strict — memento can be kept private.                               | Loose — snapshot is a regular object, readable by anyone.       |


#### When to use which?

- **OOP Way**: Use this if the object state is huge and you only need to save part of it (delta changes), or if you need strict data privacy.
- **Pythonic Way**: Use deepcopy for 90% of cases (Undo/Redo stacks, Game Saves) where the object size is manageable.

# Memento Design Pattern 

explained with a complex, real-world example: **A Vector Graphics Editor (Like Figma or Adobe Illustrator).**

#### The Scenario: Undo/Redo in a Graphics Tool

In a graphics editor, the "State" is complex. It isn't just a single string or number.
- The Canvas contains a **List of Shapes** (Circles, Rectangles).
- Each Shape has properties: **X, Y, Color, Size**.
- **The Problem**: If you move a Circle and change a Rectangle's color, the "Undo" must revert both specific objects to their previous states without mixing up references (Deep Copy issue). If you just save the list, changing a shape now would also change it in your "saved history" because they point to the same memory object.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we must implement a **Deep Clone** mechanism manually. The Memento object must detach the state completely from the current object. We often use a `clone()` or `copy()` method on every single shape to ensure the history is preserved correctly.

#### THE STATE OBJECTS (Shapes)

In [10]:
from abc import ABC, abstractmethod

class Shape(ABC):
    """
    All shapes must be cloneable so the Memento can save a specific copy.
    """
    @abstractmethod
    def clone(self) -> 'Shape':
        pass

    @abstractmethod
    def __repr__(self):
        pass

class Circle(Shape):
    def __init__(self, x, y, color):
        self.x = x
        self.y = y
        self.color = color

    def clone(self) -> 'Shape':
        return Circle(self.x, self.y, self.color)

    def __repr__(self):
        return f"Circle(x={self.x}, y={self.y}, c={self.color})"

class Rectangle(Shape):
    def __init__(self, x, y, width, height):
        self.x = x
        self.y = y
        self.width = width
        self.height = height

    def clone(self) -> 'Shape':
        return Rectangle(self.x, self.y, self.width, self.height)

    def __repr__(self):
        return f"Rect(x={self.x}, y={self.y}, w={self.width}, h={self.height})"

#### THE MEMENTO (The Snapshot)

In [11]:
class CanvasMemento:
    """
    Stores a DEEP COPY of the canvas state at a specific time.
    """
    def __init__(self, shapes: List[Shape]):
        # CRITICAL: We map(clone) the list. 
        # If we just did self.shapes = shapes, the history would corrupt.
        self._shapes = [s.clone() for s in shapes]

    def get_state(self) -> List[Shape]:
        # Return a copy again so restoring doesn't corrupt the memento
        return [s.clone() for s in self._shapes]

#### THE ORIGINATOR (The Canvas)

In [12]:
class Canvas:
    def __init__(self):
        self._shapes: List[Shape] = []

    def add_shape(self, shape: Shape):
        self._shapes.append(shape)

    def get_shapes(self):
        return self._shapes

    # --- Saving & Restoring ---
    def create_snapshot(self) -> CanvasMemento:
        print("💾 System: Creating Backup...")
        return CanvasMemento(self._shapes)

    def restore_snapshot(self, memento: CanvasMemento):
        self._shapes = memento.get_state()
        print("⏪ System: Canvas Restored.")

    def show(self):
        print(f"Canvas Content: {self._shapes}")

#### THE CARETAKER (History Manager)

In [13]:
class HistoryManager:
    def __init__(self):
        self._history: List[CanvasMemento] = []
        self._redo_stack: List[CanvasMemento] = []

    def save(self, canvas: Canvas):
        self._history.append(canvas.create_snapshot())
        self._redo_stack.clear() # New action clears redo path

    def undo(self, canvas: Canvas):
        if not self._history:
            print("❌ Nothing to Undo")
            return
        
        # Save current state to redo stack before undoing
        current_state = canvas.create_snapshot()
        self._redo_stack.append(current_state)

        # Restore previous
        memento = self._history.pop()
        canvas.restore_snapshot(memento)

#### CLIENT CODE

def main():
    canvas = Canvas()
    history = HistoryManager()

    # 1. Initial State
    c1 = Circle(10, 10, "Red")
    canvas.add_shape(c1)
    history.save(canvas) # SAVE 1

    # 2. Add Rectangle
    r1 = Rectangle(50, 50, 100, 200)
    canvas.add_shape(r1)
    
    # 3. Modify First Circle
    c1.color = "Blue" # Mutating the object
    history.save(canvas) # SAVE 2

    # 4. Delete Everything
    canvas._shapes = [] # Oops!
    canvas.show()

    # 5. Undo (Restore SAVE 2 - Circle should be Blue, Rect exists)
    history.undo(canvas)
    canvas.show()

    # 6. Undo (Restore SAVE 1 - Circle should be Red, Rect gone)
    history.undo(canvas)
    canvas.show()

if __name__ == "__main__":
    main()

## The Pythonic Way

In Python, we can skip the manual `clone()` methods for every single class. We use the powerful `copy` module (`deepcopy`). We can also implement the **Context Manager protocol** (`with` statement) to handle snapshots automatically, making the code look like a "Transaction".

#### THE STATE (Using Dataclasses)

In [16]:
from dataclasses import dataclass

@dataclass
class Shape:
    x: int
    y: int
    color: str

@dataclass
class Circle(Shape):
    radius: int

@dataclass
class Rectangle(Shape):
    width: int
    height: int

#### THE ORIGINATOR (Canvas)

In [17]:
from typing import List

class Canvas:
    def __init__(self):
        self.shapes: List[Shape] = []

    def add(self, shape):
        self.shapes.append(shape)

    def show(self):
        print(f"Canvas: {[s for s in self.shapes]}")

    # PYTHONIC SNAPSHOT
    # We don't need a specific Memento class. 
    # Any object (even a list) can be a memento if it's deep copied.
    def save(self):
        return copy.deepcopy(self.shapes)

    def restore(self, state):
        self.shapes = state

#### THE CARETAKER (Context Manager)

In [18]:
class TransactionalHistory:
    """
    A Pythonic Caretaker that acts as a wrapper.
    It allows you to group multiple changes into one 'Undo' step.
    """
    def __init__(self, target: Canvas):
        self.target = target
        self.history: List[List[Shape]] = []

    def commit(self):
        """Manually save a checkpoint"""
        print("💾 Commit: Saving state...")
        self.history.append(self.target.save())

    def undo(self):
        if not self.history:
            print("🚫 Nothing to undo.")
            return
        print("⏪ Undo Triggered.")
        previous_state = self.history.pop()
        self.target.restore(previous_state)

    # MAGIC METHOD: Context Manager
    # Allows usage: "with history.step(): do_changes()"
    def step(self):
        return self._StepContext(self)

    class _StepContext:
        def __init__(self, manager):
            self.manager = manager
        
        def __enter__(self):
            # Save state BEFORE entering the block
            self.manager.commit()
            return self.manager
        
        def __exit__(self, exc_type, exc_val, exc_tb):
            # If an error occurred inside the block, auto-rollback!
            if exc_type:
                print(f"⚠️ Error detected ({exc_val})! Rolling back...")
                self.manager.undo()
                return True # Suppress error

#### CLIENT CODE

In [19]:
def main():
    canvas = Canvas()
    history = TransactionalHistory(canvas)

    # Setup Initial State
    canvas.add(Circle(x=0, y=0, color="Red", radius=10))
    
    print("--- 1. Using 'with' block (Auto-Save) ---")
    with history.step():
        # Everything inside here is one "Step"
        canvas.add(Rectangle(x=10, y=10, color="Blue", width=50, height=50))
        # Let's modify the first circle too
        canvas.shapes[0].color = "Green"
        
        print("   (Inside Step):")
        canvas.show()

    print("\n--- 2. Outside block (State Persisted) ---")
    canvas.show()

    print("\n--- 3. Undo Operation ---")
    history.undo()
    canvas.show() # Should revert to just the Red Circle

    print("\n--- 4. Auto-Rollback on Error ---")
    try:
        with history.step():
            canvas.add(Circle(99, 99, "Black", 5))
            print("   (Added Black Circle)")
            canvas.show()
            # Simulate a crash
            raise ValueError("System Crash!")
    except:
        pass # Context manager handled it

    print("   (After Crash):")
    canvas.show() # Should NOT have the Black Circle

if __name__ == "__main__":
    main()

--- 1. Using 'with' block (Auto-Save) ---
💾 Commit: Saving state...
   (Inside Step):
Canvas: [Circle(x=0, y=0, color='Green', radius=10), Rectangle(x=10, y=10, color='Blue', width=50, height=50)]

--- 2. Outside block (State Persisted) ---
Canvas: [Circle(x=0, y=0, color='Green', radius=10), Rectangle(x=10, y=10, color='Blue', width=50, height=50)]

--- 3. Undo Operation ---
⏪ Undo Triggered.
Canvas: [Circle(x=0, y=0, color='Red', radius=10)]

--- 4. Auto-Rollback on Error ---
💾 Commit: Saving state...
   (Added Black Circle)
Canvas: [Circle(x=0, y=0, color='Red', radius=10), Circle(x=99, y=99, color='Black', radius=5)]
⚠️ Error detected (System Crash!)! Rolling back...
⏪ Undo Triggered.
   (After Crash):
Canvas: [Circle(x=0, y=0, color='Red', radius=10)]


#### Why the Pythonic version is powerful

- `copy.deepcopy`: This one function replaces the need to write `clone()` methods for `Circle`, `Rectangle`, `Triangle`, etc. It recursively copies the entire object tree, solving the "Reference Issue" instantly.
- Context Managers (`with`): This is a uniquely Pythonic feature. It allows "Transactional Editing". You can save the state automatically before a block of code runs. If the block fails (raises an Exception), the context manager can automatically call `undo()` to rollback the changes, ensuring your application never ends up in a broken state.